In [1]:
import requests
import json
import time
from datetime import datetime, timedelta, timezone

API_KEY = "..."

In [2]:
MAX_VIDEOS_PER_QUERY = 150          
MAX_COMMENTS_PER_VIDEO = 2000       
SLEEP_SECONDS = 0.2                 

In [3]:
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
COMMENTS_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

In [4]:
def rfc3339(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def yt_search_videos(query: str, published_after: str, published_before: str, max_videos: int):
    """Return list of video records: {video_id, title, description, channel, published_at}"""
    results = []
    page_token = None

    while len(results) < max_videos:
        params = {
            "part": "snippet",
            "q": query,
            "type": "video",
            "order": "date",
            "maxResults": 50,
            "publishedAfter": published_after,
            "publishedBefore": published_before,
            "key": API_KEY,
        }
        if page_token:
            params["pageToken"] = page_token
            
        res = requests.get(SEARCH_URL, params=params, timeout=30)

        if res.status_code != 200:
            print("Status code:", res.status_code)
            print("Response text:", res.text)
            return []

        data = res.json()

        for item in data.get("items", []):
            vid = item["id"].get("videoId")
            snip = item.get("snippet", {})
            if not vid:
                continue
            results.append({
                "video_id": vid,
                "query": query,
                "title": snip.get("title", ""),
                "description": snip.get("description", ""),
                "channel": snip.get("channelTitle", ""),
                "published_at": snip.get("publishedAt", ""),
            })
            if len(results) >= max_videos:
                break

        page_token = data.get("nextPageToken")
        if not page_token:
            break

        time.sleep(SLEEP_SECONDS)

    return results

In [5]:
def yt_fetch_comments(video_id: str, max_comments: int | None):
    """Return list of comments: {comment_id, text, author, like_count, published_at}"""
    comments = []
    page_token = None

    while True:
        params = {
            "part": "snippet",
            "videoId": video_id,
            "maxResults": 100,
            "textFormat": "plainText",
            "key": API_KEY,
        }
        if page_token:
            params["pageToken"] = page_token

        res = requests.get(COMMENTS_URL, params=params, timeout=30)

        if res.status_code == 403:
            return {"disabled": True, "comments": []}

        res.raise_for_status()
        data = res.json()

        for item in data.get("items", []):
            top = item["snippet"]["topLevelComment"]
            cid = top["id"]
            snip = top["snippet"]
            comments.append({
                "comment_id": cid,
                "text": snip.get("textDisplay", ""),
                "author": snip.get("authorDisplayName", ""),
                "like_count": snip.get("likeCount", 0),
                "published_at": snip.get("publishedAt", ""),
            })
            if max_comments is not None and len(comments) >= max_comments:
                return {"disabled": False, "comments": comments}

        page_token = data.get("nextPageToken")
        if not page_token:
            break

        time.sleep(SLEEP_SECONDS)

    return {"disabled": False, "comments": comments}

In [3]:
def main():

    now = datetime.now(timezone.utc)
    after = now - timedelta(days=7)

    published_after = rfc3339(after)
    published_before = rfc3339(now)

    video_map = {}  
    for q in QUERIES:
        vids = yt_search_videos(q, published_after, published_before, MAX_VIDEOS_PER_QUERY)
        for v in vids:
            video_map.setdefault(v["video_id"], v)

    videos = list(video_map.values())
    print(f"Collected {len(videos)} unique videos in last 7 days.")

    # 2) 拉评论 + 写 JSONL（流式写，避免内存爆）
    out_jsonl = "/Users/rosemary/Downloads/youtube_depression_2months.jsonl"
    all_rows = []  # 如果你想同时输出一个 json 大文件

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for i, v in enumerate(videos, 1):
            vid = v["video_id"]
            print(f"[{i}/{len(videos)}] Fetching comments for {vid} ...")

            com_res = yt_fetch_comments(vid, MAX_COMMENTS_PER_VIDEO)

            row = {
                "video": v,
                "comments_disabled": com_res["disabled"],
                "comments": com_res["comments"],
                "collected_at": rfc3339(datetime.now(timezone.utc)),
                "window": {"published_after": published_after, "published_before": published_before},
            }

            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            all_rows.append(row)

            time.sleep(SLEEP_SECONDS)

In [ ]:
    out_json = "/Users/keithleung/Documents/Columbia/Semester 2/APAN5205 Applied Machine Learning II/Project/YouTube.json'"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(all_rows, f, ensure_ascii=False, indent=2)

    print(f"\nSaved JSONL: {out_jsonl}")
    print(f"Saved JSON : {out_json}")

if __name__ == "__main__":
    main()